[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [11]:
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k

        self.router = nn.Linear(d_model, num_experts)
        experts = [nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)) for _ in range(num_experts)]
        self.experts = nn.ModuleList(experts)

    def forward(self, x):
        # route tokens to top-k experts
        B, S, d_model = x.shape

        logits = self.router(x) #(B, S, num_experts)
        top_logits, top_indices = torch.topk(logits, self.top_k, dim = -1, largest=True, sorted=True) #(B, S, top_k) sorted descending
        top_weights = torch.softmax(top_logits, dim = -1) #(B, S, top_k)

        top_indices = top_indices.reshape((B * S, -1)) #(N, top_k) --> index per expert per token
        top_weights = top_weights.reshape((B * S, -1)) #(N, top_k) --> weight per expert per token

        tokens = x.reshape((B * S), -1) #(N, d_mode) tokens
        output = torch.zeros_like(tokens) #(N, d_model)

        for expert_id, expert in enumerate(self.experts):

          token_idxs, expert_poss = torch.where(top_indices == expert_id) #(n_selected,), (n_selected,) token indices that selected expert_idx and expert pos inside the top_k in that token row in top_indices

          if token_idxs.shape[0] == 0: #no token selected this expert
            continue

          w = top_weights[token_idxs, expert_poss].unsqueeze(-1) #(n_selected, 1) to broadcast
          exp_output = expert(tokens[token_idxs]) #(n_selected, d_model)

          output[token_idxs] +=  w * exp_output #(n_selected, d_model) inserted in selected token rows --> at the end each token row will have a weighted (top_weights) sum of every expert's output

        return output.reshape((B, S, d_model))


In [12]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.0ms)
  ✅ [2/4] Has router and experts (1.2ms)
  ✅ [3/4] Router logits shape (1.6ms)
  ✅ [4/4] Gradient flow (47.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (53.3ms total)
  Progress saved. Run status() to see your dashboard.

